# Lab 3: Audio Deepfake Detection (Part A) — Traditional Methods
## Program Code Template

Welcome to Lab 3! In this assignment, we will explore how to detect AI-generated or spoofed audio (Deepfakes). Audio deepfakes often leave behind subtle acoustic artifacts that humans might not hear, but algorithms can detect.

This lab is divided into two main parts:
1. **Traditional Machine Learning Pipeline (This Notebook):** Handcrafted acoustic feature extraction followed by standard ML classifiers.
2. **Learning-Based Detection (Next Section):** Deep learning approaches (e.g., CNNs/Transformers) operating directly on spectrograms or waveforms.

**Instructions**
- Read and run the **Worked Examples** first. They show you exactly what the input and output look like.
- Cells marked `# TODO` require you to write code. Everything else is provided.


## Setup
Run the cell below to install and import the required packages.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_curve
from sklearn.datasets import make_classification

import warnings
warnings.filterwarnings('ignore')


## 1. Worked Example — Basic Feature Extraction & Classification

In traditional audio classification, we cannot feed raw varying-length audio waves directly into models like SVMs. We must extract **fixed-length feature vectors**.

Below, we simulate a 'Bona Fide' (Real) human voice and a 'Spoofed' (Fake) robotic voice using synthetic signals. We will extract basic MFCCs, take the average across all time frames, and use that as our feature.

In [ ]:
sr = 16000
DATA_DIR = "./data"

# 1. Load "Real" audio from data directory
real_file = os.path.join(DATA_DIR, "LA_T_1000137.flac")
y_real, _ = librosa.load(real_file, sr=sr)

# 2. Load "Fake" audio from data directory
# Note: Manually injected noise is removed since the real spoof tracks already contain authentic AI artifacts
fake_file = os.path.join(DATA_DIR, "LA_T_1004644.flac")
y_fake, _ = librosa.load(fake_file, sr=sr)

print("Listen to 'Real' Audio:")
display(Audio(data=y_real, rate=sr))

print("Listen to 'Fake' Audio (Notice the noise/artifacts):")
display(Audio(data=y_fake, rate=sr))

In [ ]:
# WORKED EXAMPLE: Feature Extraction
def extract_basic_feature(y, sr):
    # Extract 20 MFCCs (shape: 20 x n_frames)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

    # Average across the time axis (axis=1) to get a fixed 1D array of size 20
    mfcc_mean = np.mean(mfccs, axis=1)
    return mfcc_mean

feat_real = extract_basic_feature(y_real, sr)
feat_fake = extract_basic_feature(y_fake, sr)

print("Extracted Feature Vector Shape:", feat_real.shape)
print("Sample values (Real):", np.round(feat_real[:5], 2))
print("Sample values (Fake):", np.round(feat_fake[:5], 2))


### Training a Baseline Classifier
Now, we load real speech audio files (.flac) from the ASVspoof dataset and extract 20-dimensional MFCC feature vectors to train a baseline Support Vector Machine (SVM) classifier.

In [ ]:
# Define directory paths
DATA_DIR = "./data"
PROTOCOL_PATH = os.path.join(DATA_DIR, "ASVspoof2019.LA.cm.train.trn.txt")

# Load protocol file
df = pd.read_csv(PROTOCOL_PATH, sep=' ', header=None)
df.columns = ['speaker', 'file_name', 'system_id', 'null', 'target']

# Map labels: bona_fide -> 1 (Real), spoof -> 0 (Fake)
df['label'] = df['target'].map({'bona_fide': 1, 'spoof': 0})

# Take a small sample for quick execution
df_sample = df.sample(n=200, random_state=42)

X_list, y_list = [], []

# Extract features from audio files in the data directory
for _, row in df_sample.iterrows():
    file_path = os.path.join(DATA_DIR, f"{row['file_name']}.flac")
    if os.path.exists(file_path):
        y_audio, sr = librosa.load(file_path, sr=16000)
        mfcc_mean = np.mean(librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20), axis=1)
        X_list.append(mfcc_mean)
        y_list.append(row['label'])

X_speech = np.array(X_list)
y_speech = np.array(y_list)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_speech, y_speech, test_size=0.3, random_state=42)

# Train baseline SVM
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train, y_train)

# Evaluate
y_pred = svm_model.predict(X_test)
print(f"Real Audio Baseline Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

---
# Student TODO Tasks

The basic pipeline works, but Deepfake detection is highly challenging in the real world. To improve, we need **better features** and **domain-specific evaluation metrics**.

## TODO 1: Advanced Feature Engineering (MFCC + Deltas + Stats)
Audio deepfakes often reveal themselves in *dynamic transitions* (how the sound changes over time).

Your task is to write a function that computes `n_mfcc=24`.
Then, calculate the 1st derivative (Delta) and 2nd derivative (Delta-Delta).
Instead of just using the **mean** across time, compute both the **mean** and **standard deviation (std)** for MFCC, Delta, and Delta-Delta. Concatenate them all into a single, flat 1D NumPy array.

*Hint: The final feature vector should have a length of $24 \times 3 \times 2 = 144$.*

In [ ]:
def extract_advanced_feature(y, sr, n_mfcc=24):
    """
    Extracts Mean and Std of MFCC, Delta, and Delta-Delta.
    """
    # TODO 1.1: Extract MFCCs using librosa
    mfcc = None

    # TODO 1.2: Compute Delta and Delta-Delta using librosa.feature.delta
    delta = None
    delta2 = None

    # TODO 1.3: Compute mean and std for each across the time axis (axis=1)
    # e.g., mfcc_mean = np.mean(mfcc, axis=1)


    # TODO 1.4: Concatenate all 6 statistical arrays into one 1D array using np.hstack
    final_feature_vector = None

    return final_feature_vector

# Test your function on the simulated y_real:
# test_feat = extract_advanced_feature(y_real, sr)
# print("Advanced Feature Shape:", test_feat.shape) # Should be (144,)


## TODO 2: The Equal Error Rate (EER) Metric
In deepfake detection, datasets are extremely unbalanced (e.g., 90% fake, 10% real). **Accuracy is misleading.** Instead, the ASVspoof challenge uses **Equal Error Rate (EER)**.

EER is the point on the ROC curve where the False Positive Rate ($FPR$) equals the False Negative Rate ($FNR$).
Since $FNR = 1 - TPR$ (True Positive Rate), EER occurs where $FPR = 1 - TPR$.

Write a function to compute EER using `sklearn.metrics.roc_curve`.

In [ ]:
def compute_eer(y_true, y_scores):
    """
    Computes the Equal Error Rate (EER).

    Args:
        y_true: True binary labels (0 or 1)
        y_scores: Predicted probabilities or confidence scores for the positive class
    Returns:
        eer_value: The EER as a float percentage
    """
    # TODO 2.1: Use roc_curve to get fpr, tpr, and thresholds
    fpr, tpr, thresholds = None, None, None

    # TODO 2.2: Calculate FNR (1 - tpr)
    fnr = None

    # TODO 2.3: Find the threshold index where the absolute difference between fpr and fnr is minimized
    # Hint: Use np.nanargmin(np.absolute(fnr - fpr))
    eer_threshold_idx = None

    # TODO 2.4: The EER is the FPR (or FNR) at that specific index
    eer_value = None

    return eer_value

# Test your function with dummy data:
# dummy_y = np.array([1, 1, 0, 0, 1, 0])
# dummy_scores = np.array([0.9, 0.8, 0.4, 0.6, 0.7, 0.1])
# print("Test EER:", compute_eer(dummy_y, dummy_scores))


## TODO 3: Train and Evaluate the Best Traditional Baseline
Now, let's put it all together. We have generated a more complex tabular dataset `X_complex` representing our 144-dimensional advanced features.

Your task:
1. Initialize a `RandomForestClassifier`.
2. Fit it on the training data.
3. Predict the **probabilities** of the test data (use `.predict_proba()[:, 1]`).
4. Calculate and print the EER using your `compute_eer` function. A lower EER is better!

In [ ]:
# Generating a difficult, imbalanced mock dataset of advanced features
X_complex, y_complex = make_classification(n_samples=2000, n_features=144, n_informative=50,
                                           weights=[0.8, 0.2], random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_complex, y_complex, test_size=0.3, random_state=42)

# TODO 3.1: Initialize a RandomForestClassifier (try setting n_estimators=100)
rf_model = None

# TODO 3.2: Fit the model on (X_train_c, y_train_c)


# TODO 3.3: Get predicted probabilities for X_test_c
# Hint: my_model.predict_proba(X_test_c)[:, 1]
y_scores_rf = None

# TODO 3.4: Compute and print the EER
rf_eer = None # call compute_eer(y_test_c, y_scores_rf)
# print(f"Random Forest EER: {rf_eer:.4f}")


---
# Next Steps: Learning-Based Detection

Traditional feature engineering requires significant domain knowledge (like deciding between MFCCs or LFCCs) and throws away a lot of structural information when we average across time.

In **Part B** of this lab, we will discard handcrafted feature arrays entirely and feed the raw spectrograms (as images) or audio waveforms directly into Deep Neural Networks (CNNs) to let the model *learn* the best features automatically.